In [0]:
%run ./00_Organizacao_do_Ambiente

bronze_schema: workspace.bronze
silver_schema: workspace.silver
gold_schema: workspace.gold
landing_path: /Volumes/workspace/rocket/cinedata_raw


In [0]:
from pyspark.sql import functions as F, Row
from pyspark.sql.window import Window
from datetime import datetime

dq_results = []

def dq_check(nome_tabela: str, nome_check: str, df, condicao):
    total = df.count()
    falhas = df.filter(~condicao).count()
    passou = falhas == 0
    dq_results.append(Row(table_name=nome_tabela, check_name=nome_check, total_rows=total,
                           failed_rows=falhas, passed=passou, checked_at=datetime.now()))
    print(f"[{'PASS' if passou else 'FAIL'}] {nome_tabela} | {nome_check} | {falhas}/{total} falharam")

def dq_check_unique(nome_tabela: str, nome_check: str, df, colunas_chave: list):
    total = df.count()
    duplicadas = df.groupBy(*colunas_chave).count().filter("count > 1").count()
    passou = duplicadas == 0
    dq_results.append(Row(table_name=nome_tabela, check_name=nome_check, total_rows=total,
                           failed_rows=duplicadas, passed=passou, checked_at=datetime.now()))
    print(f"[{'PASS' if passou else 'FAIL'}] {nome_tabela} | {nome_check} | {duplicadas} chaves duplicadas de {total}")

In [0]:
df_dim_movies = (
    spark.table(f"{silver_schema}.tb_info_filmes")
    # surrogate key artificial, desacoplada do id_filme de origem
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")))
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse")
)

dq_check_unique("gold.dim_movies", "sk_movie_id único", df_dim_movies, ["sk_movie_id"])
dq_check_unique("gold.dim_movies", "id_filme único", df_dim_movies, ["id_filme"])

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"gold.dim_movies: {df_dim_movies.count()} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_movies | sk_movie_id único | 0 chaves duplicadas de 97879
[PASS] gold.dim_movies | id_filme único | 0 chaves duplicadas de 97879
gold.dim_movies: 97879 linhas


In [0]:
df_dim_genres = (
    spark.table(f"{silver_schema}.tb_generos")
    .select("nome_genero")
    .distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")))
    .select("sk_genre_id", "nome_genero")
)

dq_check_unique("gold.dim_genres", "sk_genre_id único", df_dim_genres, ["sk_genre_id"])
dq_check_unique("gold.dim_genres", "nome_genero único", df_dim_genres, ["nome_genero"])

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"gold.dim_genres: {df_dim_genres.count()} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_genres | sk_genre_id único | 0 chaves duplicadas de 19
[PASS] gold.dim_genres | nome_genero único | 0 chaves duplicadas de 19


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_genres: 19 linhas


In [0]:
df_dim_people = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    # dedup por (nome, papel): a mesma pessoa pode ter papéis diferentes em filmes diferentes
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

dq_check_unique("gold.dim_people", "sk_person_id único", df_dim_people, ["sk_person_id"])
dq_check("gold.dim_people", "tipo_pessoa em domínio válido", df_dim_people,
         F.col("tipo_pessoa").isin(["Ator", "Diretor", "Roteirista"]))

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"gold.dim_people: {df_dim_people.count()} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_people | sk_person_id único | 0 chaves duplicadas de 415699
[PASS] gold.dim_people | tipo_pessoa em domínio válido | 0/415699 falharam
gold.dim_people: 415699 linhas


In [0]:
df_dim_companies = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")))
    .select("sk_company_id", "nome_produtora")
)

dq_check_unique("gold.dim_companies", "sk_company_id único", df_dim_companies, ["sk_company_id"])
dq_check_unique("gold.dim_companies", "nome_produtora único", df_dim_companies, ["nome_produtora"])

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"gold.dim_companies: {df_dim_companies.count()} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_companies | sk_company_id único | 0 chaves duplicadas de 43197
[PASS] gold.dim_companies | nome_produtora único | 0 chaves duplicadas de 43197
gold.dim_companies: 43197 linhas


In [0]:
df_dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

df_dim_reviews = (
    spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        # avg() ignora nulos automaticamente (notas fora de 0-10 não distorcem a média)
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )
    .join(df_dim_movies_lk, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

dq_check_unique("gold.dim_reviews", "sk_review_id único", df_dim_reviews, ["sk_review_id"])
dq_check_unique("gold.dim_reviews", "sk_movie_id único (1 registro por filme)", df_dim_reviews, ["sk_movie_id"])

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"gold.dim_reviews: {df_dim_reviews.count()} linhas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.dim_reviews | sk_review_id único | 0 chaves duplicadas de 27303
[PASS] gold.dim_reviews | sk_movie_id único (1 registro por filme) | 0 chaves duplicadas de 27303
gold.dim_reviews: 27303 linhas


In [0]:
df_movies_lancados = (
    spark.table(f"{gold_schema}.dim_movies")
    # Atividade exige que a fato consolide só filmes lançados
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

df_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento")

df_fact_movies_performance = (
    df_movies_lancados
    # financeiro e métricas já são 1:1 por id_filme desde a Silver -> grão preservado sem esforço extra
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)

dq_check_unique("gold.fact_movies_performance", "sk_movie_id único (grão = 1 por filme)",
                 df_fact_movies_performance, ["sk_movie_id"])

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact: {df_fact_movies_performance.count()} linhas (esperado == {df_movies_lancados.count()} filmes lançados)")

[PASS] gold.fact_movies_performance | sk_movie_id único (grão = 1 por filme) | 0 chaves duplicadas de 96463
fact: 96463 linhas (esperado == 96463 filmes lançados)


In [0]:
df_dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
df_dim_genres_lk = spark.table(f"{gold_schema}.dim_genres")
df_dim_people_lk = spark.table(f"{gold_schema}.dim_people")
df_dim_companies_lk = spark.table(f"{gold_schema}.dim_companies")

# inner join com dim_movies naturalmente restringe a bridge aos filmes existentes no modelo
df_bridge_movie_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .join(df_dim_movies_lk, on="id_filme", how="inner")
    .join(df_dim_genres_lk, on="nome_genero", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)
df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

df_bridge_movie_person = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
    .join(df_dim_movies_lk, on="id_filme", how="inner")
    .join(df_dim_people_lk,
          (F.col("nome_entidade") == df_dim_people_lk["nome_pessoa"]) &
          (F.col("tipo_entidade") == df_dim_people_lk["tipo_pessoa"]),
          how="inner")
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)
df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

df_bridge_movie_company = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(df_dim_movies_lk, on="id_filme", how="inner")
    .join(df_dim_companies_lk, F.col("nome_entidade") == df_dim_companies_lk["nome_produtora"], how="inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)
df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print(f"bridge_movie_genre: {df_bridge_movie_genre.count()}")
print(f"bridge_movie_person: {df_bridge_movie_person.count()}")
print(f"bridge_movie_company: {df_bridge_movie_company.count()}")

bridge_movie_genre: 140437
bridge_movie_person: 759263
bridge_movie_company: 114890


In [0]:
# Base: metadados do filme que vêm direto da dim_movies (título, ano, sinopse)
df_dim_movies_ctx = spark.table(f"{gold_schema}.dim_movies").select(
    "sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse"
)

# Financeiro vem da fato -- só filmes "Lançado" têm linha aqui (join left cobre o resto como nulo)
df_fact_ctx = spark.table(f"{gold_schema}.fact_movies_performance").select(
    "sk_movie_id", "receita_usd", "orcamento_usd"
)

df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_dim_people_ctx = spark.table(f"{gold_schema}.dim_people")

# Um filme tem vários atores (várias linhas na bridge) -- precisamos agregar em UMA string por filme.
# collect_list junta os nomes numa lista, array_join transforma a lista em texto separado por vírgula.
df_atores = (
    df_bridge_person.join(df_dim_people_ctx, on="sk_person_id")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.array_join(F.collect_list("nome_pessoa"), ", ").alias("atores_principais"))
)

# Mesma lógica pro diretor -- às vezes tem mais de um (co-direção), por isso agregamos igual aos atores
df_diretores = (
    df_bridge_person.join(df_dim_people_ctx, on="sk_person_id")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.array_join(F.collect_list("nome_pessoa"), ", ").alias("diretor"))
)

df_genai_context = (
    df_dim_movies_ctx
    # left join: nem todo filme tem financeiro (só "Lançado") nem elenco/diretor (column shift pode ter zerado)
    .join(df_fact_ctx, on="sk_movie_id", how="left")
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")

    # Resolvendo campo por campo aqui, garantimos que a frase final NUNCA fica nula.
    .withColumn("ano_texto", F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")))
    .withColumn("receita_texto", F.when(F.col("receita_usd").isNotNull(),
                                         F.concat(F.lit("US$ "), F.format_number("receita_usd", 2)))
                                   .otherwise(F.lit("valor não divulgado")))
    .withColumn("orcamento_texto", F.when(F.col("orcamento_usd").isNotNull(),
                                           F.concat(F.lit("US$ "), F.format_number("orcamento_usd", 2)))
                                     .otherwise(F.lit("valor não divulgado")))
    .withColumn("atores_texto", F.coalesce(F.col("atores_principais"), F.lit("elenco não divulgado")))
    .withColumn("diretor_texto", F.coalesce(F.col("diretor"), F.lit("diretor não divulgado")))
        .withColumn("sinopse_texto", F.coalesce(
        F.trim(F.regexp_replace(F.col("sinopse"), r'[\\"]+', ' ')),
        F.lit("sinopse não disponível")
    ))

    # Frase corrida seguindo o template exato pedido na atividade, usando só as colunas _texto
    .withColumn("llm_context_document",
        F.concat(
            F.lit("O filme "), F.col("titulo"),
            F.lit(", lançado no ano de "), F.col("ano_texto"),
            F.lit(", faturou "), F.col("receita_texto"),
            F.lit(" e teve um custo de "), F.col("orcamento_texto"),
            F.lit(". Estrelado por "), F.col("atores_texto"),
            F.lit(" e dirigido por "), F.col("diretor_texto"),
            F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse_texto"),
            F.lit(".")
        )
    )
    # movie_id/title são os nomes de coluna exigidos pelo PDF (rastreabilidade até a dim_movies)
    .select(F.col("id_filme").alias("movie_id"), F.col("titulo").alias("title"), "llm_context_document")
)

# Prova de que o tratamento de nulos funcionou: essa checagem tem que dar PASS (0 falhas)
dq_check("gold.gold_genai_movies_context", "llm_context_document nunca nulo", df_genai_context,
         F.col("llm_context_document").isNotNull())

df_genai_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_genai_movies_context")

print(f"gold_genai_movies_context: {df_genai_context.count()} linhas")
display(df_genai_context.limit(3))

[PASS] gold.gold_genai_movies_context | llm_context_document nunca nulo | 0/97879 falharam
gold_genai_movies_context: 97879 linhas


movie_id,title,llm_context_document
14564,Rings,"O filme Rings, lançado no ano de 2017, faturou US$ 83,080,890.00 e teve um custo de US$ 25,000,000.00. Estrelado por Patrick Walker, Chuck David Willis, Bonnie Morgan, Vincent D'Onofrio, Aimee Teegarden, Johnny Galecki, Laura Slade Wiggins, Zach Roerig, Alex Roe, Matilda Lutz e dirigido por F. Javier Gutiérrez, o filme possui a seguinte sinopse: Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a movie within the movie that no one has ever seen before. First you watch it. Then you die.."
32471,Mixtape,"O filme Mixtape, lançado no ano de 2021, faturou valor não divulgado e teve um custo de valor não divulgado. Estrelado por Lucas Yao, Audrey Hsieh, Olga Petsa, Nick Thune, Jackson Rathbone, Gemma Brooke Allen, Julie Bowen, Anthony Timpano, Kiefer O'Reilly e dirigido por Valerie Weiss, o filme possui a seguinte sinopse: sinopse não disponível."
38258,Grizzly II: Revenge,"O filme Grizzly II: Revenge, lançado no ano de 2020, faturou valor não divulgado e teve um custo de US$ 7,500,000.00. Estrelado por elenco não divulgado e dirigido por diretor não divulgado, o filme possui a seguinte sinopse: All hell breaks loose when a giant grizzly."
